<a href="https://www.kaggle.com/code/nihalabhay/chest-imagenet?scriptVersionId=340761541" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== VERIFY: list the actual filenames in the single-source weights dataset =====
import os
CHEST_WEIGHTS_DIR = '/kaggle/input/datasets/nihalabhay/chestsinglesourcebest/'
files = sorted(os.listdir(CHEST_WEIGHTS_DIR))
for f in files:
    print(f)
print(f"\nTotal files: {len(files)}")

ss_custom_chex_chest.keras
ss_custom_chex_log.csv
ss_custom_chex_log_retry.csv
ss_custom_nih_chest.keras
ss_custom_nih_log.csv
ss_custom_vinbig_chest.keras
ss_custom_vinbig_log.csv
ss_eff_chex_best.keras
ss_eff_chex_phase1.keras
ss_eff_chex_phase1_log.csv
ss_eff_chex_phase2_log.csv
ss_eff_nih_best.keras
ss_eff_nih_phase1.keras
ss_eff_nih_phase1_log.csv
ss_eff_nih_phase2_log.csv
ss_eff_vinbig_best.keras
ss_eff_vinbig_phase1.keras
ss_eff_vinbig_phase1_log.csv
ss_eff_vinbig_phase2_log.csv
ss_mob_chex_best.keras
ss_mob_chex_phase1.keras
ss_mob_chex_phase1_log.csv
ss_mob_chex_phase2_log.csv
ss_mob_nih_best.keras
ss_mob_nih_phase1.keras
ss_mob_nih_phase1_log.csv
ss_mob_nih_phase2_log.csv
ss_mob_vinbig_best.keras
ss_mob_vinbig_phase1.keras
ss_mob_vinbig_phase1_log.csv
ss_mob_vinbig_phase2_log.csv
ss_res_chex_best.keras
ss_res_chex_phase1.keras
ss_res_chex_phase1_log.csv
ss_res_chex_phase2_log.csv
ss_res_nih_best.keras
ss_res_nih_phase1.keras
ss_res_nih_phase1_log.csv
ss_res_nih_phase2_log.csv

In [3]:
# ===== STAGE 12 SESSION: FULL REBUILD + CROSS-SOURCE 3x3 MATRIX (same columns as skin's Stage 12 CSV) =====
!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import glob
import tensorflow as tf
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import load_model
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

CHEX_DIR      = '/kaggle/input/datasets/ashery/chexpert/'
CHEX_CSV      = '/kaggle/input/datasets/ashery/chexpert/train.csv'
NIH_DIR       = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/'
NIH_CSV       = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv'
VINBIG_CSV    = '/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv'
VINBIG_PNG    = '/kaggle/input/datasets/xhlulu/vinbigdata-chest-xray-png-512px-original-ratio/train/'
CHEST_WEIGHTS_DIR = '/kaggle/input/datasets/nihalabhay/chestsinglesourcebest/'
FINAL_CLASSES = ['no_finding', 'pathology']
IMG_SIZE   = 224
BATCH_SIZE = 32
SUBSAMPLE_SEED = 42
TARGET_PER_SOURCE = 15000

chex = pd.read_csv(CHEX_CSV)
chex['label'] = np.where(chex['No Finding'] == 1.0, 'no_finding', 'pathology')
chex['patient_id'] = chex['Path'].str.extract(r'(patient\d+)')
chex['image_path'] = CHEX_DIR + chex['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)
chex['source'] = 'chex'

nih = pd.read_csv(NIH_CSV)
nih['label'] = np.where(nih['Finding Labels'] == 'No Finding', 'no_finding', 'pathology')
nih['patient_id'] = nih['Patient ID'].astype(str)
nih['source'] = 'nih'
nih_files = glob.glob(os.path.join(NIH_DIR, '**', '*.png'), recursive=True)
nih_map = {os.path.basename(p): p for p in nih_files}
nih['image_path'] = nih['Image Index'].map(nih_map)

vin_raw = pd.read_csv(VINBIG_CSV)
img_findings = vin_raw.groupby('image_id')['class_id'].apply(lambda s: set(s))
vin = pd.DataFrame({'image_id': img_findings.index})
vin['label'] = img_findings.apply(lambda fs: 'no_finding' if fs == {14} else 'pathology').values
vin['image_path'] = VINBIG_PNG + vin['image_id'] + '.png'
vin['source'] = 'vinbig'; vin['patient_id'] = None

def subsample_by_patient(df, target_n, seed=SUBSAMPLE_SEED):
    pats = df['patient_id'].drop_duplicates().sample(frac=1.0, random_state=seed).tolist()
    sizes = df['patient_id'].value_counts()
    chosen, count = [], 0
    for p in pats:
        n = sizes[p]
        if count + n > target_n and count >= target_n * 0.98: break
        chosen.append(p); count += n
        if count >= target_n: break
    return df[df['patient_id'].isin(chosen)].reset_index(drop=True)

chex_s = subsample_by_patient(chex, TARGET_PER_SOURCE)
nih_s  = subsample_by_patient(nih,  TARGET_PER_SOURCE)
vin_s  = vin.copy()

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, unstratified fallback. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['label'].agg(lambda s: s.value_counts().index[0]).reset_index()
    p_tr, p_tmp = safe_split(pg, 'label', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'label', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    return pick(p_tr), pick(p_va), pick(p_te)

def split_image_level(df, rs=SEED, tag="VinBig"):
    tr, tmp = safe_split(df, 'label', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'label', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

c_tr, c_va, c_te = split_patient_level(chex_s, tag="CheXpert")
n_tr, n_va, n_te = split_patient_level(nih_s,  tag="NIH")
v_tr, v_va, v_te = split_image_level(vin_s)

test_sets = {'chex': c_te, 'nih': n_te, 'vinbig': v_te}
for src, d in test_sets.items():
    print(f"{src} test set: {len(d):,} images")

def make_eval_gen(df, preprocess_fn):
    if preprocess_fn is None:
        idg = ImageDataGenerator(rescale=1./255)
    else:
        idg = ImageDataGenerator(preprocessing_function=preprocess_fn)
    return idg.flow_from_dataframe(df, x_col='image_path', y_col='label',
                                    target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                                    class_mode='categorical', classes=FINAL_CLASSES,
                                    color_mode='rgb', shuffle=False)

def weight_path(arch, src):
    if arch == 'custom':
        return f'{CHEST_WEIGHTS_DIR}ss_custom_{src}_chest.keras'
    else:
        return f'{CHEST_WEIGHTS_DIR}ss_{arch}_{src}_best.keras'

arch_specs = {'custom': None, 'eff': eff_pre, 'mob': mob_pre, 'res': res_pre}
arch_display = {'custom': 'Custom CNN', 'eff': 'EfficientNetB0', 'mob': 'MobileNetV2', 'res': 'ResNet50'}

results = []
sources = ['chex', 'nih', 'vinbig']

for arch, preproc in arch_specs.items():
    for train_src in sources:
        model_path = weight_path(arch, train_src)
        print(f"\nLoading {model_path}")
        m = load_model(model_path)
        for test_src in sources:
            gen = make_eval_gen(test_sets[test_src], preproc)
            preds = m.predict(gen, verbose=0)
            proba = preds[:, 1]
            y = np.array(gen.classes)
            yhat = (proba >= 0.5).astype(int)

            auc = roc_auc_score(y, proba)
            acc = accuracy_score(y, yhat)
            f1 = f1_score(y, yhat, average='macro')   # matches skin's macro_f1 computation method
            cm = confusion_matrix(y, yhat, labels=[0,1])
            tn, fp, fn, tp = cm.ravel()
            sens = tp/(tp+fn) if (tp+fn) > 0 else float('nan')
            spec = tn/(tn+fp) if (tn+fp) > 0 else float('nan')
            setting = 'within' if train_src == test_src else 'cross'   # skin's naming, replaces chest's 'diagonal' boolean

            print(f"  {arch_display[arch]} trained-on-{train_src} -> tested-on-{test_src} [{setting}]: "
                  f"n={len(y)} AUC={auc:.4f} F1={f1:.4f} Acc={acc:.4f} Sens={sens:.4f} Spec={spec:.4f}")

            results.append({
                'arch': arch_display[arch],
                'train_source': train_src,
                'test_source': test_src,
                'setting': setting,
                'n': len(y),
                'macro_auc': auc,          # binary task: this is the single AUC value, not averaged across classes,
                'macro_f1': f1,            # unlike skin there are only 2 classes so no macro-averaging is actually
                'accuracy': acc,           # happening for auc/sens/spec, the column names are kept aligned to skin's
                'macro_sensitivity': sens, # file for direct comparability, the underlying number is a plain binary
                'macro_specificity': spec, # metric in every case here
            })
        del m
        import gc; gc.collect()

results_df = pd.DataFrame(results)
results_df.to_csv('/kaggle/working/chest_stage12_crossmatrix.csv', index=False)
print("\n\nSaved chest_stage12_crossmatrix.csv, columns match skin's Stage 12 CSV")
print(results_df.to_string(index=False))

# ---- Off-diagonal AUC drop, mean AND median, per architecture ----
print("\n\n=== OFF-DIAGONAL (cross) AUC DROP PER ARCHITECTURE ===")
for arch in arch_display.values():
    sub = results_df[results_df['arch'] == arch]
    within_auc = sub[sub['setting'] == 'within'].set_index('train_source')['macro_auc']
    cross = sub[sub['setting'] == 'cross']
    drops = [within_auc[row['train_source']] - row['macro_auc'] for _, row in cross.iterrows()]
    print(f"{arch}: mean drop = {np.mean(drops):.4f} | median drop = {np.median(drops):.4f} | "
          f"min cell AUC = {cross['macro_auc'].min():.4f} | max cell AUC = {cross['macro_auc'].max():.4f}")

Seed 42 set, TF 2.19.0, tf.keras module: tensorflow.keras
chex test set: 2,223 images
nih test set: 2,032 images
vinbig test set: 2,250 images

Loading /kaggle/input/datasets/nihalabhay/chestsinglesourcebest/ss_custom_chex_chest.keras
Found 2223 validated image filenames belonging to 2 classes.
  Custom CNN trained-on-chex -> tested-on-chex [within]: n=2223 AUC=0.7980 F1=0.5803 Acc=0.8934 Sens=0.9849 Spec=0.1375
Found 2032 validated image filenames belonging to 2 classes.
  Custom CNN trained-on-chex -> tested-on-nih [cross]: n=2032 AUC=0.5629 F1=0.5008 Acc=0.5108 Sens=0.7560 Spec=0.3247
Found 2250 validated image filenames belonging to 2 classes.
  Custom CNN trained-on-chex -> tested-on-vinbig [cross]: n=2250 AUC=0.6710 F1=0.4852 Acc=0.4853 Sens=0.8012 Spec=0.3545

Loading /kaggle/input/datasets/nihalabhay/chestsinglesourcebest/ss_custom_nih_chest.keras
Found 2223 validated image filenames belonging to 2 classes.
  Custom CNN trained-on-nih -> tested-on-chex [cross]: n=2223 AUC=0.763